In [ ]:
import pandas as pd
import numpy as np

In [6]:
data = pd.read_csv("../data/books.csv")
print(data.shape)
data.head()


(52478, 25)


,bookId,title,series,author,rating,description,language,isbn,genres,characters,...,firstPublishDate,awards,numRatings,ratingsByStars,likedPercent,setting,coverImg,bbeScore,bbeVotes,price
0,2767052-the-hunger-games,The Hunger Games,The Hunger Games #1,Suzanne Collins,4.33,WINNING MEANS FAME AND FORTUNE.LOSING MEANS CE...,English,9780439023481,"['Young Adult', 'Fiction', 'Dystopia', 'Fantas...","['Katniss Everdeen', 'Peeta Mellark', 'Cato (H...",...,NaN,['Locus Award Nominee for Best Young Adult Boo...,6376780,"['3444695', '1921313', '745221', '171994', '93...",96.0,"['District 12, Panem', 'Capitol, Panem', 'Pane...",https://i.gr-assets.com/images/S/compressed.ph...,2993816,30516,5.09
1,2.Harry_Potter_and_the_Order_of_the_Phoenix,Harry Potter and the Order of the Phoenix,Harry Potter #5,"J.K. Rowling, Mary GrandPré (Illustrator)",4.50,There is a door at the end of a silent corrido...,English,9780439358071,"['Fantasy', 'Young Adult', 'Fiction', 'Magic',...","['Sirius Black', 'Draco Malfoy', 'Ron Weasley'...",...,06/21/03,['Bram Stoker Award for Works for Young Reader...,2507623,"['1593642', '637516', '222366', '39573', '14526']",98.0,['Hogwarts School of Witchcraft and Wizardry (...,https://i.gr-assets.com/images/S/compressed.ph...,2632233,26923,7.38
2,2657.To_Kill_a_Mockingbird,To Kill a Mockingbird,To Kill a Mockingbird,Harper Lee,4.28,The unforgettable novel of a childhood in a sl...,English,9999999999999,"['Classics', 'Fiction', 'Historical Fiction', ...","['Scout Finch', 'Atticus Finch', 'Jem Finch', ...",...,07/11/60,"['Pulitzer Prize for Fiction (1961)', 'Audie A...",4501075,"['2363896', '1333153', '573280', '149952', '80...",95.0,"['Maycomb, Alabama (United States)']",https://i.gr-assets.com/images/S/compressed.ph...,2269402,23328,NaN
3,1885.Pride_and_Prejudice,Pride and Prejudice,NaN,"Jane Austen, Anna Quindlen (Introduction)",4.26,Alternate cover edition of ISBN 9780679783268S...,English,9999999999999,"['Classics', 'Fiction', 'Romance', 'Historical...","['Mr. Bennet', 'Mrs. Bennet', 'Jane Bennet', '...",...,01/28/13,[],2998241,"['1617567', '816659', '373311', '113934', '767...",94.0,"['United Kingdom', 'Derbyshire, England (Unite...",https://i.gr-assets.com/images/S/compressed.ph...,1983116,20452,NaN
4,41865.Twilight,Twilight,The Twilight Saga #1,Stephenie Meyer,3.60,About three things I was absolutely positive.\...,English,9780316015844,"['Young Adult', 'Fantasy', 'Romance', 'Vampire...","['Edward Cullen', 'Jacob Black', 'Laurent', 'R...",...,10/05/05,"['Georgia Peach Book Award (2007)', 'Buxtehude...",4964519,"['1751460', '1113682', '1008686', '542017', '5...",78.0,"['Forks, Washington (United States)', 'Phoenix...",https://i.gr-assets.com/images/S/compressed.ph...,1459448,14874,2.1


In [3]:
data.columns

Index(['bookId', 'title', 'series', 'author', 'rating', 'description',
       'language', 'isbn', 'genres', 'characters', 'bookFormat', 'edition',
       'pages', 'publisher', 'publishDate', 'firstPublishDate', 'awards',
       'numRatings', 'ratingsByStars', 'likedPercent', 'setting', 'coverImg',
       'bbeScore', 'bbeVotes', 'price'],
      dtype='str')

In [16]:
data.isna().sum()

bookId                  0
title                   0
series              29008
author                  0
rating                  0
description          1338
language             3806
isbn                    0
genres                  0
characters              0
bookFormat           1473
edition             47523
pages                2347
publisher            3696
publishDate           880
firstPublishDate    21326
awards                  0
numRatings              0
ratingsByStars          0
likedPercent          622
setting                 0
coverImg              605
bbeScore                0
bbeVotes                0
price               14365
dtype: int64

In [10]:
data['description'].str.len().describe()

count    51140.000000
mean       861.412828
std        548.797120
min          3.000000
25%        520.000000
50%        792.000000
75%       1087.000000
max      24733.000000
Name: description, dtype: float64

In [13]:
data['rating'].describe()

count    52478.000000
mean         4.021878
std          0.367146
min          0.000000
25%          3.820000
50%          4.030000
75%          4.230000
max          5.000000
Name: rating, dtype: float64

In [14]:
data['author'].describe()

count                               52478
unique                              28227
top       Nora Roberts (Goodreads Author)
freq                                   86
Name: author, dtype: object

In [27]:
data['pages'].describe()

count     50131
unique     1365
top         320
freq       1049
Name: pages, dtype: object

Hidden missing values ('[]')

In [38]:
print((data['genres'] == '[]').sum())
print((data['characters'] == '[]').sum())
print((data['setting'] == '[]').sum())
print((data['isbn'] == '9999999999999').sum())  # placeholder ISBN
print((data['isbn'] == '0000000000000').sum())

4623
38712
40900
4354
0


### Duplicate values
With these present, recommender may return the same book twice
Duplicate descriptions are often different editions or placeholder text

In [17]:
data['bookId'].duplicated().sum()

np.int64(54)

In [18]:
data.duplicated(['title', 'author']).sum()

np.int64(88)

In [19]:
data['description'].dropna().duplicated().sum()

np.int64(252)

### Popularity and rating reliability
Number of ratings is very skewed. Have to consider weighting/normalization, minimum ratings filter,...

In [21]:
data['numRatings'].describe()

count    5.247800e+04
mean     1.787865e+04
std      1.039448e+05
min      0.000000e+00
25%      3.410000e+02
50%      2.307000e+03
75%      9.380500e+03
max      7.048471e+06
Name: numRatings, dtype: float64

In [23]:
(data['numRatings'] < 10).sum()

np.int64(2444)

In [25]:
(data['numRatings'] == 0).sum()

np.int64(71)

In [26]:
(data['rating'] == 0).sum()

np.int64(71)

### Genre distribution

In [43]:
import ast # because of '[]' type of missing values, treating lists as lists and not as strings

genres = data['genres'].apply(lambda x: ast.literal_eval(x))
genres.explode().value_counts().head(30)

genres
Fiction                    31638
Romance                    15495
Fantasy                    15046
Young Adult                11869
Contemporary               10520
Nonfiction                  8251
Adult                       8246
Novels                      7805
Mystery                     7702
Historical Fiction          7665
Audiobook                   7307
Classics                    6902
Adventure                   6452
Historical                  6383
Paranormal                  6030
Literature                  5836
Science Fiction             5374
Childrens                   5226
Thriller                    4587
Magic                       4248
Humor                       4227
History                     3685
Crime                       3675
Contemporary Romance        3624
Suspense                    3474
Urban Fantasy               3458
Middle Grade                3389
Chick Lit                   3358
Science Fiction Fantasy     3302
Supernatural                3196
Nam

In [44]:
genres.str.len().describe()  # genres per book

count    52478.000000
mean         7.769313
std          3.578427
min          0.000000
25%          6.000000
50%         10.000000
75%         10.000000
max         10.000000
Name: genres, dtype: float64

### Metadata to potentially filter on
- language
- pages
- firstPublishedDate / publishDate
- author